In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/rechecked-finaldatasetforcausal/finaldatasetwithcourseinformationforcausalanalysis.csv


In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from tqdm import tqdm
import re

print("Loading data...")
df = pd.read_csv('/kaggle/input/rechecked-finaldatasetforcausal/finaldatasetwithcourseinformationforcausalanalysis.csv')

def clean_name(name):
    return re.sub(r'[^a-zA-Z0-9]', '_', name)

Loading data...


In [4]:
# combine sentences based on comment id(id_x)
df_grouped = df.groupby('id_x').agg({
    'rate': 'mean',
    'group_category': 'first',
    'predicted_aspect': lambda x: list(set(x)),
    'predicted_sentiment_ensemble': lambda x: list(x),
    'price': 'first',
    'is_paid': 'first',
    'num_lectures': 'first'
}).reset_index()
df_grouped

,id_x,rate,group_category,predicted_aspect,predicted_sentiment_ensemble,price,is_paid,num_lectures
0,133,3.0,Technology and IT,[Design & Material],[Neutral],74.99,True,56.0
1,135,4.0,Technology and IT,[Design & Material],[Positive],74.99,True,56.0
2,157,4.0,Business and Finance,"[Financial, Skill & Career]","[Neutral, Neutral]",39.99,True,6.0
3,180,5.0,Business and Finance,"[Design & Material, Skill & Career, Financial]","[Positive, Positive, Positive, Positive, Posit...",149.99,True,87.0
4,258,4.0,Creative and Design,"[Design & Material, Instructor, Platform & Tec...","[Positive, Neutral, Neutral]",19.99,True,33.0
...,...,...,...,...,...,...,...,...
799877,126708292,3.5,Technology and IT,[Skill & Career],[Positive],19.99,True,15.0
799878,126708298,5.0,Business and Finance,[Assignment & Tutorial],"[Positive, Positive]",19.99,True,0.0
799879,126708538,1.0,Business and Finance,[Instructor],[Negative],149.99,True,97.0
799880,126708818,5.0,Technology and IT,[Platform & Technical],[Neutral],199.99,True,46.0


In [5]:

aspects = df['predicted_aspect'].unique()
for aspect in tqdm(aspects, desc="Creating sentiment columns"):
    clean_aspect = clean_name(aspect)
    #if there are more than one specific aspect, i will use mean value.
    df_grouped[f'sentiment_{clean_aspect}'] = df_grouped.apply(
        lambda row: np.mean([2 if s == 'Positive' else 1 if s == 'Neutral' else 0 
                             for a, s in zip(row['predicted_aspect'], row['predicted_sentiment_ensemble']) if a == aspect]) 
        if aspect in row['predicted_aspect'] else np.nan, 
        axis=1
    )

Creating sentiment columns: 100%|██████████| 6/6 [01:19<00:00, 13.24s/it]


In [6]:
df_grouped

,id_x,rate,group_category,predicted_aspect,predicted_sentiment_ensemble,price,is_paid,num_lectures,sentiment_Instructor,sentiment_Assignment___Tutorial,sentiment_Design___Material,sentiment_Skill___Career,sentiment_Financial,sentiment_Platform___Technical
0,133,3.0,Technology and IT,[Design & Material],[Neutral],74.99,True,56.0,NaN,NaN,1.0,NaN,NaN,NaN
1,135,4.0,Technology and IT,[Design & Material],[Positive],74.99,True,56.0,NaN,NaN,2.0,NaN,NaN,NaN
2,157,4.0,Business and Finance,"[Financial, Skill & Career]","[Neutral, Neutral]",39.99,True,6.0,NaN,NaN,NaN,1.0,1.0,NaN
3,180,5.0,Business and Finance,"[Design & Material, Skill & Career, Financial]","[Positive, Positive, Positive, Positive, Posit...",149.99,True,87.0,NaN,NaN,2.0,2.0,2.0,NaN
4,258,4.0,Creative and Design,"[Design & Material, Instructor, Platform & Tec...","[Positive, Neutral, Neutral]",19.99,True,33.0,1.0,NaN,2.0,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
799877,126708292,3.5,Technology and IT,[Skill & Career],[Positive],19.99,True,15.0,NaN,NaN,NaN,2.0,NaN,NaN
799878,126708298,5.0,Business and Finance,[Assignment & Tutorial],"[Positive, Positive]",19.99,True,0.0,NaN,2.0,NaN,NaN,NaN,NaN
799879,126708538,1.0,Business and Finance,[Instructor],[Negative],149.99,True,97.0,0.0,NaN,NaN,NaN,NaN,NaN
799880,126708818,5.0,Technology and IT,[Platform & Technical],[Neutral],199.99,True,46.0,NaN,NaN,NaN,NaN,NaN,1.0


In [7]:
df_grouped['is_free'] = ~df_grouped['is_paid']

In [8]:
def exact_matching(df, target_aspect, aspects):
    def create_match_key(row):
        other_aspects = sorted([asp for asp in aspects if asp != target_aspect])
        sentiments = []
        for asp in other_aspects:
            value = row[f'sentiment_{clean_name(asp)}']
            if pd.isna(value):
                sentiments.append(f"{asp}:NaN")
            else:
                sentiments.append(f"{asp}:{value:.1f}")
        
        price_group = pd.cut([row['price']], bins=[0, 19.99, 49.99, 139.99, np.inf], 
                             labels=['Low', 'Medium', 'High', 'Premium'])[0]
        lectures_group = pd.cut([row['num_lectures']], bins=[0, 17, 32, 60, np.inf], 
                                labels=['Few', 'Some', 'Many', 'Extensive'])[0]
        
        sentiments.extend([
            f"is_free:{row['is_free']}",
            f"price_group:{price_group}",
            f"lectures_group:{lectures_group}"
        ])
        return '_'.join(sentiments)

    df['match_key'] = df.apply(create_match_key, axis=1)
    
    clean_target = clean_name(target_aspect)
    group_effects = []
    group_weights = []

    for _, group in tqdm(df.groupby('match_key'), desc=f"Exact matching for {target_aspect}", leave=False):
        # ตัดข้อมูลที่ aspect ที่เราสนใจเป็น NaN ออก
        group = group.dropna(subset=[f'sentiment_{clean_target}'])
        
        sentiments = group[f'sentiment_{clean_target}'].unique()
        if len(sentiments) > 1:
            sentiment_ratings = {s: group[group[f'sentiment_{clean_target}'] == s]['rate'].mean() for s in sentiments}
            
            effects = []
            for i in range(len(sentiments) - 1):
                for j in range(i+1, len(sentiments)):
                    s1, s2 = sentiments[i], sentiments[j]
                    effect = (sentiment_ratings[s2] - sentiment_ratings[s1]) / (s2 - s1)
                    effects.append(effect)
            
            if effects:
                group_ate = np.mean(effects)
                group_effects.append(group_ate)
                group_weights.append(len(group))

    if group_effects:
        weighted_ate = np.average(group_effects, weights=group_weights)
        unweighted_ate = np.mean(group_effects)
        t_stat, p_value = stats.ttest_1samp(group_effects, 0)
        return {
            'aspect': target_aspect,
            'weighted_ate': weighted_ate,
            'unweighted_ate': unweighted_ate,
            'p_value': p_value,
            'effect_range': (min(group_effects), max(group_effects)),
            'num_groups': len(group_effects),
            'total_samples': sum(group_weights)
        }
    return {
        'aspect': target_aspect,
        'weighted_ate': 0,
        'unweighted_ate': 0,
        'p_value': 1,
        'effect_range': (0, 0),
        'num_groups': 0,
        'total_samples': 0
    }


In [9]:
def analyze_exact_matching_by_category(df, aspects):
    categories = df['group_category'].unique()
    all_results = []
    
    for category in categories:
        df_category = df[df['group_category'] == category]
        
        results = []
        for aspect in aspects:
            result = exact_matching(df_category, aspect, aspects)
            result['category'] = category
            results.append(result)
        
        results_df = pd.DataFrame(results)
        print(f"\nExact Matching Results for {category}:")
        print(results_df.to_string(index=False))
        
        all_results.append(results_df)
    
    # Combine all results
    combined_results = pd.concat(all_results, ignore_index=True)
    return combined_results

# ทำการวิเคราะห์
exact_matching_results = analyze_exact_matching_by_category(df_grouped, aspects)

print("\nExact Matching analysis complete.")

# บันทึกผลลัพธ์เป็นไฟล์ CSV
exact_matching_results.to_csv('exact_matching_results_by_category.csv', index=False)

/tmp/ipykernel_35/2085417865.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['match_key'] = df.apply(create_match_key, axis=1)
/tmp/ipykernel_35/2085417865.py:24: SettingWithCopyWarning:                        
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['match_key'] = df.apply(create_match_key, axis=1)
/tmp/ipykernel_35/2085417865.py:24: SettingWithCopyWarning:                                   
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead




Exact Matching Results for Technology and IT:
               aspect  weighted_ate  unweighted_ate       p_value              effect_range  num_groups  total_samples          category
           Instructor      1.187156        0.650198 4.756017e-140               (-3.0, 4.0)        1352         130875 Technology and IT
Assignment & Tutorial      1.058958        0.609700  1.003512e-96 (-2.166666666666667, 4.0)        1048          54042 Technology and IT
    Design & Material      1.079069        0.658860 1.440238e-132               (-3.0, 4.0)        1126          64613 Technology and IT
       Skill & Career      0.938167        0.562432  1.161457e-78               (-2.0, 4.0)         810          34031 Technology and IT
            Financial      1.090842        0.654374  1.334094e-68               (-3.0, 4.0)         694          13880 Technology and IT
 Platform & Technical      1.040520        0.600173 1.860748e-103               (-4.0, 3.5)        1122          49281 Technology a

/tmp/ipykernel_35/2085417865.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['match_key'] = df.apply(create_match_key, axis=1)
/tmp/ipykernel_35/2085417865.py:24: SettingWithCopyWarning:                        
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['match_key'] = df.apply(create_match_key, axis=1)
/tmp/ipykernel_35/2085417865.py:24: SettingWithCopyWarning:                                   
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead




Exact Matching Results for Business and Finance:
               aspect  weighted_ate  unweighted_ate      p_value effect_range  num_groups  total_samples             category
           Instructor      1.128267        0.647901 1.085161e-70  (-2.5, 3.5)         660          51521 Business and Finance
Assignment & Tutorial      0.959710        0.637027 1.856928e-55  (-2.0, 4.0)         486          22141 Business and Finance
    Design & Material      1.056880        0.676713 2.457208e-76 (-2.0, 3.75)         607          36186 Business and Finance
       Skill & Career      0.889942        0.643104 3.559173e-57  (-3.5, 4.0)         524          27983 Business and Finance
            Financial      1.004885        0.708889 9.113583e-57  (-4.0, 4.0)         470          13564 Business and Finance
 Platform & Technical      1.029059        0.662218 1.105856e-51  (-3.0, 4.5)         462          11823 Business and Finance


/tmp/ipykernel_35/2085417865.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['match_key'] = df.apply(create_match_key, axis=1)
/tmp/ipykernel_35/2085417865.py:24: SettingWithCopyWarning:                        
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['match_key'] = df.apply(create_match_key, axis=1)
/tmp/ipykernel_35/2085417865.py:24: SettingWithCopyWarning:                                   
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead




Exact Matching Results for Creative and Design:
               aspect  weighted_ate  unweighted_ate      p_value effect_range  num_groups  total_samples            category
           Instructor      1.080226        0.637472 2.469107e-77  (-2.0, 3.5)         637          47678 Creative and Design
Assignment & Tutorial      0.933962        0.638462 1.321473e-47  (-3.0, 4.0)         447          17516 Creative and Design
    Design & Material      0.967771        0.640917 9.422790e-59  (-4.0, 4.0)         595          23470 Creative and Design
       Skill & Career      0.869178        0.596477 6.909516e-43  (-3.5, 3.5)         451          15774 Creative and Design
            Financial      0.930729        0.672023 1.293487e-41  (-2.5, 3.5)         384           6781 Creative and Design
 Platform & Technical      0.972977        0.694471 5.385296e-49  (-2.5, 4.0)         433           9819 Creative and Design


... (4 outputs left out)

/tmp/ipykernel_35/2085417865.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['match_key'] = df.apply(create_match_key, axis=1)
/tmp/ipykernel_35/2085417865.py:24: SettingWithCopyWarning:                        
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['match_key'] = df.apply(create_match_key, axis=1)
/tmp/ipykernel_35/2085417865.py:24: SettingWithCopyWarning:                                   
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead




Exact Matching Results for Marketing and Productivity:
               aspect  weighted_ate  unweighted_ate      p_value              effect_range  num_groups  total_samples                   category
           Instructor      1.139513        0.684110 2.888613e-56               (-3.0, 3.0)         428          26631 Marketing and Productivity
Assignment & Tutorial      1.042037        0.671490 8.596493e-34               (-1.5, 4.5)         283           9321 Marketing and Productivity
    Design & Material      1.088616        0.770662 3.728133e-59               (-4.0, 3.5)         436          21412 Marketing and Productivity
       Skill & Career      0.971173        0.615974 2.648421e-32              (-1.75, 4.0)         302          10958 Marketing and Productivity
            Financial      1.070037        0.711832 5.008911e-33 (-4.5, 2.666666666666667)         285           5557 Marketing and Productivity
 Platform & Technical      1.049781        0.673954 2.094586e-37          

/tmp/ipykernel_35/2085417865.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['match_key'] = df.apply(create_match_key, axis=1)
/tmp/ipykernel_35/2085417865.py:24: SettingWithCopyWarning:                        
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['match_key'] = df.apply(create_match_key, axis=1)
/tmp/ipykernel_35/2085417865.py:24: SettingWithCopyWarning:                                   
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead




Exact Matching Results for Health and Fitness:
               aspect  weighted_ate  unweighted_ate      p_value effect_range  num_groups  total_samples           category
           Instructor      0.970861        0.711189 2.376385e-31  (-1.5, 4.0)         266          13668 Health and Fitness
Assignment & Tutorial      0.873586        0.638289 3.187267e-23  (-1.0, 3.5)         199           9233 Health and Fitness
    Design & Material      1.049247        0.805366 2.865192e-34  (-2.5, 4.0)         240           7688 Health and Fitness
       Skill & Career      0.886539        0.567374 6.370225e-16  (-1.5, 4.5)         149           4881 Health and Fitness
            Financial      1.053743        0.832279 5.751536e-21  (-2.0, 3.5)         127           1616 Health and Fitness
 Platform & Technical      0.961499        0.551406 4.650064e-10  (-4.0, 3.0)         128           1907 Health and Fitness

Exact Matching analysis complete.
